In [10]:
from apscheduler.schedulers.background import BackgroundScheduler
import time
from datetime import datetime
import pandas as pd
import numpy as np
import pickle
import threading

class GPC_runner:
    def __init__(self):
        self.iter = 0
        self.GPCResults = None
        self.GPCscheduler = BackgroundScheduler()
        self.event = threading.Event()
        self.RealTime = [0,0,0]
        
    def GPC_inject(self):
        print("Injection number", str(self.iter+1), "time=", datetime.now().strftime("%H:%M:%S"))
        self.RealTime[self.iter] = datetime.now()
        self.iter +=1
        if self.iter >= 3:
            self.GPCscheduler.remove_job('GPC_inject')
            self.StartTime = [1,108,216]
            threading.Timer(3, self.delayed_return).start() #3s delay

    def delayed_return(self):
        print("Data collected")
        self.event.set()  # Unblocks the waiting function

    

    def GPC_analysis(self,GPCresults, StartTime):
        # Define detection limit for Peak filtering
        detectlimit = 7
        peaknum = 3
        # Initialise Results dictionary and other storage arrays
        chrom = np.zeros((154,6))
        polpeak = pd.DataFrame()

        Results = {"rawchrom": GPCresults,
                "Mn": np.zeros(peaknum),
                "Mw": np.zeros(peaknum),
                "PD": np.zeros(peaknum),
                "MP": np.zeros(peaknum),
                "chrom": [],
                "polpeak": [],
                "StartTime": StartTime,
                "Averages":[],
                }

        # Load calibration polynomial
        with open('G:\My Drive\Coding\GPC\Calibration.pkl', 'rb') as f:
            poly = pickle.load(f)
        # Extract coefficients
        coeffs = poly.coeffs
        #Convert GPC results to two separate arrays
        elution_time, RI_signal = GPCresults.to_numpy().T
        #Add start time manipulation (see MATLAB)
        #Points on baseline (ADJUST after calibration)

        fitpoints = (
            (elution_time > 0) & (elution_time < 110) |
            (elution_time > 279) & (elution_time < 282) |
            (elution_time > 450) & (elution_time < 453) |
            (elution_time > 680)
        )

        # Fit polynomial to baseline
        baseline = np.polyfit(elution_time[fitpoints], RI_signal[fitpoints], 1)
        # Subtract baseline
        RI_signal -= np.polyval(baseline, elution_time)
        for i in range(peaknum):
            # Define region of interest (i.e. chromatogram) for analysis in each loop & filter
            elution_time_ana = elution_time[StartTime[i]:StartTime[i]+154]
            RI_signal_ana = RI_signal[StartTime[i]:StartTime[i]+154]
            # Make elution_time_ana start at 0
            elution_time_ana = elution_time_ana - elution_time_ana[0]

            # Filter for data between min and max calibrant elution
            peakfilter = ((elution_time_ana > 122) & (elution_time_ana < 201))
            peakdata = RI_signal_ana[peakfilter]
            peaktime = elution_time_ana[peakfilter]

            # Check if peak exists - if not, set penalty values
            if max(peakdata) < 0.25:
                PD = 3
                MP = 0
                Mn = 0
                chrom[:,2*i] = elution_time_ana
                chrom[:,2*i+1] = RI_signal_ana
                # polpeak = 0
            else:
                # Normalise peak data
                peakdata = peakdata/max(peakdata)
                # Set detection limit relative to peak height and eliminate data below threshold
                peakfilter = peakdata > max(peakdata)/detectlimit
                peakdata = peakdata[peakfilter]
                peaktime = peaktime[peakfilter]
                # Convert time scale to MW using calibration polynomial
                M1 = coeffs[0]*peaktime**3
                M2 = coeffs[1]*peaktime**2
                M3 = coeffs[2]*peaktime
                M4 = coeffs[3]
                MWData = M1+M2+M3+M4
                MWData = 10**MWData
            
                # Calculate number average molecular weight
                Mn = sum(peakdata*MWData)/sum(peakdata)
                Mw = sum(peakdata*MWData**2)/sum(peakdata*MWData)
                PD = Mw/Mn
                # Find location of peak maximum 
                MP = MWData[np.argmax(peakdata)]
                # Store chromatogram data
                chrom[:,2*i] = elution_time_ana
                chrom[:,(2*i)+ 1] = RI_signal_ana
                # Store polynomial peak data
                newpolpeak = pd.DataFrame([MWData,peakdata]).T
                polpeak = pd.concat([polpeak,newpolpeak],axis=1).fillna(0)
            # Store results in Results dict:
            Results["Mn"][i] = Mn
            Results["Mw"][i] = Mw
            Results["PD"][i] = PD
            Results["MP"][i] = MP
            
        Results["chrom"] = chrom
        Results["polpeak"] = polpeak
        Results["Averages"] = pd.DataFrame([[np.mean(Results["Mn"]), np.mean(Results["Mw"]), np.mean(Results["PD"]), np.mean(Results["MP"])]], columns=["Mn", "Mw", "PD", "MP"])
        return Results

    def GPC_run(self):
        print("GPC triggered!")
        self.GPCscheduler.add_job(self.GPC_inject, 'interval', seconds=0.5, id='GPC_inject')
        self.GPCscheduler.start()
        self.event.wait()
        GPCdata = pd.read_csv('GPCres.csv')
        StartTime = self.StartTime
        self.GPCResults = self.GPC_analysis(GPCdata, StartTime)
        print("GPC completed! Results =")
        print(f"{self.GPCResults['Averages']}") 
        print(self.RealTime)
        return self.GPCResults

GPCrunner = GPC_runner()
GPCresults = GPCrunner.GPC_run()


GPC triggered!
Injection number 1 time= 13:25:16
Injection number 2 time= 13:25:17
Injection number 3 time= 13:25:17
Data collected
GPC completed! Results =
             Mn            Mw        PD            MP
0  29155.388688  39477.292537  1.354578  35302.731484
[datetime.datetime(2025, 3, 20, 13, 25, 16, 775105), datetime.datetime(2025, 3, 20, 13, 25, 17, 269548), datetime.datetime(2025, 3, 20, 13, 25, 17, 770828)]


In [41]:
Results = GPCrunner.GPCResults["Averages"]


In [ ]:
import pandas as pd
import pickle
import plotly.graph_objects as go

fig = go.Figure()
fig1 = go.Figure()
GPCresults = pd.read_csv('GPCres.csv')
StartTime = [1,108,216]
# Define detection limit for Peak filtering
detectlimit = 7
peaknum = 3
# Initialise Results dictionary and other storage arrays
chrom = np.zeros((154,6))
polpeak = pd.DataFrame()

Results = {"rawchrom": GPCresults,
           "Mn": np.zeros(peaknum),
           "Mw": np.zeros(peaknum),
           "PD": np.zeros(peaknum),
           "MP": np.zeros(peaknum),
           "chrom": [],
           "polpeak": [],
           "StartTime": StartTime}

# Load calibration polynomial
with open('Calibration.pkl', 'rb') as f:
    poly = pickle.load(f)
# Extract coefficients
coeffs = poly.coeffs
#Convert GPC results to two separate arrays
elution_time, RI_signal = GPCresults.to_numpy().T
#Add start time manipulation (see MATLAB)
#Points on baseline (ADJUST after calibration)

fitpoints = (
    (elution_time > 0) & (elution_time < 110) |
    (elution_time > 279) & (elution_time < 282) |
    (elution_time > 450) & (elution_time < 453) |
    (elution_time > 680)
)

# Fit polynomial to baseline
baseline = np.polyfit(elution_time[fitpoints], RI_signal[fitpoints], 1)
# Subtract baseline
RI_signal -= np.polyval(baseline, elution_time)
for i in range(peaknum):
    # Define region of interest (i.e. chromatogram) for analysis in each loop & filter
    elution_time_ana = elution_time[StartTime[i]:StartTime[i]+154]
    RI_signal_ana = RI_signal[StartTime[i]:StartTime[i]+154]
    # Make elution_time_ana start at 0
    elution_time_ana = elution_time_ana - elution_time_ana[0]

    # Filter for data between min and max calibrant elution
    peakfilter = ((elution_time_ana > 122) & (elution_time_ana < 201))
    peakdata = RI_signal_ana[peakfilter]
    peaktime = elution_time_ana[peakfilter]

    # Check if peak exists - if not, set penalty values
    if max(peakdata) < 0.25:
        PD = 3
        MP = 0
        Mn = 0
        Mw = 0
        chrom[:,2*i] = elution_time_ana
        chrom[:,2*i+1] = RI_signal_ana
        # polpeak = 0
    else:
        # Normalise peak data
        peakdata = peakdata/max(peakdata)
        # Set detection limit relative to peak height and eliminate data below threshold
        peakfilter = peakdata > max(peakdata)/detectlimit
        peakdata = peakdata[peakfilter]
        peaktime = peaktime[peakfilter]
        # Convert time scale to MW using calibration polynomial
        M1 = coeffs[0]*peaktime**3
        M2 = coeffs[1]*peaktime**2
        M3 = coeffs[2]*peaktime
        M4 = coeffs[3]
        MWData = M1+M2+M3+M4
        MWData = 10**MWData
       
        # Calculate number average molecular weight
        Mn = sum(peakdata*MWData)/sum(peakdata)
        Mw = sum(peakdata*MWData**2)/sum(peakdata*MWData)
        PD = Mw/Mn
        # Find location of peak maximum 
        MP = MWData[np.argmax(peakdata)]
        # Store chromatogram data
        chrom[:,2*i] = elution_time_ana
        chrom[:,(2*i)+ 1] = RI_signal_ana
        # Store polynomial peak data
        newpolpeak = pd.DataFrame([MWData,peakdata]).T
        polpeak = pd.concat([polpeak,newpolpeak],axis=1).fillna(0)
    # Store results in Results dict:
    Results["Mn"][i] = Mn
    Results["Mw"][i] = Mw
    Results["PD"][i] = PD
    Results["MP"][i] = MP
        
Results["chrom"] = chrom
Results["polpeak"] = polpeak

        
        




In [73]:
Results["polpeak"]

,0,1,0,1,0,1
0,86785.326774,0.190130,78612.301919,0.221954,89796.591708,0.146619
1,65216.531680,0.467263,66896.203029,0.360822,76101.363409,0.299406
2,55850.262557,0.683075,57374.303754,0.537831,64760.577127,0.474523
3,48252.639845,0.828919,49463.178630,0.740448,55567.981746,0.657004
4,41880.023023,0.971042,42948.507010,0.873768,47973.919689,0.809791
5,36579.229250,1.000000,37453.950685,0.963327,41707.831780,0.955211
6,32088.756961,0.964258,32900.150590,1.000000,36428.814611,1.000000
7,28340.840424,0.878190,29004.692233,0.961911,32006.981521,0.959273
8,25130.916209,0.822676,25718.778314,0.867217,25864.679889,0.862802
9,22414.834576,0.738331,22916.227717,0.789085,23047.766688,0.814632


In [76]:
fig1.show()

In [ ]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=elution_time, y=RI_signal, mode='lines', name='GPC'))
fig.show()

fig.add_trace(go.Scatter(x=elution_time_ana, 
                            y=RI_signal_ana,
                            mode = 'lines',
                            name = f"Run {i+1}"
                        )
    )

fig1.add_trace(go.Scatter(x=MWData, 
                        y=peakdata,
                        mode = 'lines',
                        name = f"Run {i+1}"
                    )
) 

In [63]:
i = 0
(2*i)+ 1

1

In [29]:
elution_time_ana = elution_time[StartTime[i]:StartTime[i]+154]